Empezamos el proceso de entender y calificar los datos

In [11]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

print("Librerías importadas.")

# --- Carga de datos de muestra ---
notebook_dir = Path.cwd()

# SUBIR UN NIVEL para llegar a CODEABLELABS-ET, luego bajar a transactions
file_path = notebook_dir.parent / 'transactions' / 'transactions_20251025_204217.csv'

print(f"Buscando en: {file_path}")

try:
    df_raw = pd.read_csv(file_path)
    print(f"✓ Datos cargados exitosamente")
    display(df_raw.head())
    
except FileNotFoundError:
    print(f"❌ No encontrado: {file_path}")

Librerías importadas.
Buscando en: e:\pruebatecnica\CodeableLabs-ET\transactions\transactions_20251025_204217.csv
✓ Datos cargados exitosamente


,transaction_id,user_id,merchant_id,amount,currency,status,timestamp,payment_method,payment_provider,country,response_code,response_message,fee_percentage,transaction_fee,net_amount,device_type,ip_address,user_agent,attempt_number,processing_time_ms,three_ds_verified,installments,category,is_international,settlement_date
0,TXN00000001,6719,735,14.89,CLP,approved,2025-08-09 13:43:08,debit_card,Visa Debit,CL,00,Transaction approved,3.46,0.52,14.37,desktop,79.165.146.105,Mozilla/5.0 (X11; Linux x86_64; rv:1.9.7.20) G...,1,1491,True,1,retail,False,2025-08-11
1,TXN00000002,4754,465,14.29,MXN,approved,2025-09-11 11:56:53,credit_card,Diners Club,MX,00,Transaction approved,2.97,0.42,13.87,mobile,164.232.190.236,Mozilla/5.0 (Windows; U; Windows 98; Win 9x 4....,1,2275,True,1,other,False,2025-09-12
2,TXN00000003,2912,586,78.59,USD,approved,2025-09-30 05:46:34,ewallet,Rappi Pay,MX,00,Transaction approved,3.01,2.37,76.22,mobile,58.127.96.196,Opera/8.79.(Windows NT 10.0; ht-HT) Presto/2.9...,1,2010,NaN,1,other,True,2025-10-02
3,TXN00000004,9235,650,281.73,CLP,declined,2025-09-01 06:36:57,debit_card,Visa Debit,CL,65,Security violation,0.00,0.00,0.00,mobile,142.16.101.255,Mozilla/5.0 (iPad; CPU iPad OS 5_1_1 like Mac ...,3,2344,True,1,retail,False,NaN
4,TXN00000005,8637,677,30.44,CLP,approved,2025-08-09 03:31:39,ewallet,MercadoPago,CL,00,Transaction approved,3.52,1.07,29.37,api,164.162.187.154,API/1.0,1,1920,NaN,1,entertainment,False,2025-08-10


In [12]:
# ===================================================
# 2. DIAGNÓSTICO: NULOS Y TIPOS DE DATOS
# ===================================================

print("--- 2.1 Información General (Tipos de Datos y Nulos) ---")
print("Revisamos los tipos de datos (Dtype) y el conteo de no-nulos (Non-Null Count).")
print("Esto nos dice qué columnas debemos convertir (ej. 'timestamp' a fecha).")

# .info() es el mejor resumen para esto
df_raw.info()

print("\n\n--- 2.2 Conteo Específico de Valores Nulos ---")

# Calculamos los nulos
null_counts = df_raw.isnull().sum()

# Filtramos para mostrar solo columnas que SÍ tienen nulos
null_counts_with_nulls = null_counts[null_counts > 0].sort_values(ascending=False)

if null_counts_with_nulls.empty:
    print("¡Excelente! No se encontraron valores nulos en este lote.")
else:
    print("Se encontraron valores nulos en las siguientes columnas:")
    
    # Calculamos el porcentaje
    total_rows = len(df_raw)
    null_percentage = (null_counts_with_nulls / total_rows) * 100
    
    # Creamos un mini-dataframe para mostrarlo bonito
    null_summary = pd.DataFrame({
        'Cantidad de Nulos': null_counts_with_nulls,
        'Porcentaje (%)': null_percentage.round(2)
    })
    
    # display() lo formatea como una tabla HTML en el notebook
    display(null_summary)

--- 2.1 Información General (Tipos de Datos y Nulos) ---
Revisamos los tipos de datos (Dtype) y el conteo de no-nulos (Non-Null Count).
Esto nos dice qué columnas debemos convertir (ej. 'timestamp' a fecha).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   transaction_id      100 non-null    object 
 1   user_id             100 non-null    int64  
 2   merchant_id         100 non-null    int64  
 3   amount              100 non-null    float64
 4   currency            100 non-null    object 
 5   status              100 non-null    object 
 6   timestamp           100 non-null    object 
 7   payment_method      100 non-null    object 
 8   payment_provider    100 non-null    object 
 9   country             100 non-null    object 
 10  response_code       100 non-null    object 
 11  response_message    100 non-null    object 
 1

,Cantidad de Nulos,Porcentaje (%)
three_ds_verified,32,32.0
settlement_date,16,16.0
ip_address,1,1.0


In [13]:
# ===================================================
# 3. PROCESO DE LIMPIEZA Y TRANSFORMACIÓN
# ===================================================

print("Iniciando el proceso de limpieza...")

# 1. Crear una copia para no modificar el original (Buena práctica)
df_clean = df_raw.copy()
print(f"DataFrame copiado. {len(df_clean)} filas iniciales.")


# 2. Manejar duplicados (Requisito de la prueba)
# Nos basamos en 'transaction_id' como clave única.
initial_rows = len(df_clean)
df_clean.drop_duplicates(subset=['transaction_id'], keep='first', inplace=True)
final_rows = len(df_clean)
print(f"- [PASO 1] Duplicados eliminados: {initial_rows - final_rows} filas.")


# 3. Manejar valores nulos (según nuestro diagnóstico)
# Rellenamos 'three_ds_verified' nulos con False (asumimos no verificado)
df_clean['three_ds_verified'] = df_clean['three_ds_verified'].fillna(False)
# Rellenamos 'ip_address' nulos con 'Unknown'
df_clean['ip_address'] = df_clean['ip_address'].fillna('Unknown')
print("- [PASO 2] Valores nulos rellenados (three_ds_verified, ip_address).")


# 4. Validar y convertir tipos de datos (Fechas)
# Convertimos 'timestamp' a datetime
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
# Convertimos 'settlement_date' a datetime. 
# 'errors=coerce' convierte los nulos (de transacciones 'declined') en NaT (Not a Time).
df_clean['settlement_date'] = pd.to_datetime(df_clean['settlement_date'], errors='coerce')
print("- [PASO 3] Columnas de fecha convertidas a datetime.")


# 5. Validar y convertir tipos de datos (Booleanos)
# Aseguramos que 'three_ds_verified' (que ya no tiene nulos) sea booleano
df_clean['three_ds_verified'] = df_clean['three_ds_verified'].astype(bool)
# 'is_international' ya era bool, pero lo re-aseguramos
df_clean['is_international'] = df_clean['is_international'].astype(bool)
print("- [PASO 4] Columnas de flags convertidas a booleano.")


# 6. Estandarizar formatos de texto (Requisito de la prueba)
text_cols_to_lower = ['status', 'payment_method', 'category', 'device_type']
text_cols_to_upper = ['currency', 'country']

for col in text_cols_to_lower:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].str.lower()
        
for col in text_cols_to_upper:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].str.upper()

print("- [PASO 5] Formatos de texto estandarizados (mayúsculas/minúsculas).")


# ===================================================
# 4. VERIFICACIÓN POST-LIMPIEZA
# ===================================================

print("\n--- 4.1 Verificación de Tipos y Nulos (Post-Limpieza) ---")
# Volvemos a ejecutar .info() en el DataFrame limpio
# Ahora 'three_ds_verified' e 'ip_address' deben tener 100 non-null
# Y 'timestamp' / 'settlement_date' deben ser 'datetime64[ns]'
df_clean.info()


print("\n\n--- 4.2 Muestra de Datos Limpios ---")
# Mostramos la cabecera para ver los cambios (ej. 'CLP' en mayúsculas, 'approved' en minúsculas)
display(df_clean.head())

Iniciando el proceso de limpieza...
DataFrame copiado. 100 filas iniciales.
- [PASO 1] Duplicados eliminados: 0 filas.
- [PASO 2] Valores nulos rellenados (three_ds_verified, ip_address).
- [PASO 3] Columnas de fecha convertidas a datetime.
- [PASO 4] Columnas de flags convertidas a booleano.
- [PASO 5] Formatos de texto estandarizados (mayúsculas/minúsculas).

--- 4.1 Verificación de Tipos y Nulos (Post-Limpieza) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   transaction_id      100 non-null    object        
 1   user_id             100 non-null    int64         
 2   merchant_id         100 non-null    int64         
 3   amount              100 non-null    float64       
 4   currency            100 non-null    object        
 5   status              100 non-null    object        
 6   timestamp           1

,transaction_id,user_id,merchant_id,amount,currency,status,timestamp,payment_method,payment_provider,country,response_code,response_message,fee_percentage,transaction_fee,net_amount,device_type,ip_address,user_agent,attempt_number,processing_time_ms,three_ds_verified,installments,category,is_international,settlement_date
0,TXN00000001,6719,735,14.89,CLP,approved,2025-08-09 13:43:08,debit_card,Visa Debit,CL,00,Transaction approved,3.46,0.52,14.37,desktop,79.165.146.105,Mozilla/5.0 (X11; Linux x86_64; rv:1.9.7.20) G...,1,1491,True,1,retail,False,2025-08-11
1,TXN00000002,4754,465,14.29,MXN,approved,2025-09-11 11:56:53,credit_card,Diners Club,MX,00,Transaction approved,2.97,0.42,13.87,mobile,164.232.190.236,Mozilla/5.0 (Windows; U; Windows 98; Win 9x 4....,1,2275,True,1,other,False,2025-09-12
2,TXN00000003,2912,586,78.59,USD,approved,2025-09-30 05:46:34,ewallet,Rappi Pay,MX,00,Transaction approved,3.01,2.37,76.22,mobile,58.127.96.196,Opera/8.79.(Windows NT 10.0; ht-HT) Presto/2.9...,1,2010,False,1,other,True,2025-10-02
3,TXN00000004,9235,650,281.73,CLP,declined,2025-09-01 06:36:57,debit_card,Visa Debit,CL,65,Security violation,0.00,0.00,0.00,mobile,142.16.101.255,Mozilla/5.0 (iPad; CPU iPad OS 5_1_1 like Mac ...,3,2344,True,1,retail,False,NaT
4,TXN00000005,8637,677,30.44,CLP,approved,2025-08-09 03:31:39,ewallet,MercadoPago,CL,00,Transaction approved,3.52,1.07,29.37,api,164.162.187.154,API/1.0,1,1920,False,1,entertainment,False,2025-08-10
